# Autograd from scratch — Notebook 1

You are going to build a working automatic differentiation engine, then use it
to train a neural network. Everything here runs on plain Python — no PyTorch,
no installs.

**How this works**

- Read a short section, then fill in the `YOUR CODE HERE` parts.
- Run the **CHECK** cell right after it. It either prints `PASS` or tells you
  exactly what's wrong.
- Section 3 creates a cell labelled **THE VALUE CLASS**. You will keep coming
  back to that one cell and adding methods to it. After each edit, re-run it,
  then re-run the check.
- Stop at any `PASS` — every checkpoint is a real milestone.

There is no reading longer than a couple of paragraphs. If you get stuck for
more than ~10 minutes on any exercise, that's a signal to ask, not to grind.

---
## 1. A derivative is a slope you can just measure

Forget symbolic differentiation for a second. A derivative answers: *if I nudge
this input a tiny bit, how much does the output move?* You can measure that
directly — evaluate the function, nudge, evaluate again, divide.

The accurate version straddles the point instead of stepping forward from it:

$$\frac{\partial f}{\partial x_i} \approx \frac{f(\dots x_i + h \dots) - f(\dots x_i - h \dots)}{2h}$$

Nudge **one** input at a time, holding all others fixed — that is exactly what
"partial derivative" means.

**Exercise.** Implement `numerical_grad(f, inputs, h)`. It takes a function of
several arguments and a list of input values, and returns a list of partial
derivatives, one per input. Use the formula above. `h = 1e-5` is a good default.

You will actually use this later as the ground truth to test your real engine
against, so it is worth getting right.

In [ ]:
def numerical_grad(f, inputs, h=1e-5):
    '''Estimate the gradient of f at `inputs` by central differences.

    f      : a function taking len(inputs) floats and returning one float
    inputs : list of floats
    returns: list of floats, same length as inputs
    '''
    grads = []
    for i in range(len(inputs)):
        # YOUR CODE HERE:
        #   build two copies of `inputs`, one with inputs[i] + h and one
        #   with inputs[i] - h, call f on each, and append the estimate.
        pass
    return grads

In [ ]:
# --- CHECK 1 ---
import math

def _f(a, b, c):
    return -a**3 + math.sin(3*b) - 1.0/c + b**2.5 - a**0.5

_expected = [-12.353553390593273, 10.25699027111255, 0.0625]
_got = numerical_grad(_f, [2.0, 3.0, 4.0])
assert _got is not None and len(_got) == 3, "should return a list of 3 numbers"
for _i, (_g, _e) in enumerate(zip(_got, _expected)):
    assert abs(_g - _e) < 1e-5, f"dim {_i}: expected {_e}, got {_g}"

def _g2(x, y):
    return x*y + x
assert abs(numerical_grad(_g2, [3.0, -2.0])[0] - (-1.0)) < 1e-6
print("PASS - you can measure gradients without calculus")

---
## 2. Why that isn't good enough

Your `numerical_grad` is correct, and it is useless for training.

It costs **two evaluations of the entire function per input**. A small network
has thousands of parameters; a real one has millions. One training step would
mean millions of forward passes. There is also a precision problem: `f(x+h)` and
`f(x-h)` are nearly equal floats, so subtracting them throws away most of your
significant digits.

What we want instead: run the function **once**, and get the derivative with
respect to *every* input in one sweep. To do that, the program has to remember
what operations it performed and in what order — it needs to build a graph.

That is what the next section starts.

---
## 3. The `Value` object — recording the forward pass

We wrap every number in an object that stores, alongside the number itself, the
operation that produced it and the values it came from. Then `d = a*b + c` does
not just compute a number — it leaves behind a graph we can walk later.

Each `Value` holds:

- `data` — the number
- `grad` — the derivative of the final output with respect to *this* value.
  Starts at `0.0`; you will fill it in from Section 5 onward.
- `_prev` — the Values this one was built from
- `_op` — a label, purely for debugging
- `_backward` — a function; for now a no-op, filled in later

**Exercise.** In the cell below, implement `__add__` and `__mul__`. Each one
should return a **new** `Value` whose `data` is the result, and whose `_prev`
records the two inputs.

This cell is **THE VALUE CLASS** — you will return to it repeatedly. Keep it as
your single source of truth and re-run it after every edit.

In [ ]:
# ============================ THE VALUE CLASS ============================
# Keep editing THIS cell as you work through the notebook. Re-run it after
# every change, then re-run whatever CHECK you are working on.

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None      # filled in from Section 5
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # ---------------------------------------------------------- Section 3
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # YOUR CODE HERE: build the output Value, then return it.
        raise NotImplementedError

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # YOUR CODE HERE
        raise NotImplementedError

    # ---------------------------------------------------------- Section 6
    # def backward(self):
    #     ...

    # ---------------------------------------------------------- Section 7
    # def tanh(self): ...
    # def exp(self):  ...
    # def __pow__(self, k): ...

    # ------------------------------- free conveniences (already done) ----
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)

    # uncomment these once __pow__ works (Section 7):
    # def __truediv__(self, other):  return self * other**-1
    # def __rtruediv__(self, other): return other * self**-1

In [ ]:
# --- CHECK 3 ---
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b
d = e + c

assert isinstance(e, Value) and isinstance(d, Value), "ops must return Value objects"
assert abs(e.data - (-6.0)) < 1e-9, f"a*b should be -6.0, got {e.data}"
assert abs(d.data - 4.0) < 1e-9, f"e+c should be 4.0, got {d.data}"
assert e._prev == {a, b}, "e should remember it came from a and b"
assert d._prev == {e, c}, "d should remember it came from e and c"
assert abs((a * 2).data - 4.0) < 1e-9, "multiplying by a plain number should work"
assert abs((2 + a).data - 4.0) < 1e-9, "plain number on the left should work"
print("PASS - your Values build a graph")

---
## 4. One hop of the chain rule, by hand

Before automating it, do it manually once. Consider:

```
x = 3.0
y = -2.0
p = x * y
q = p + x
L = q * 2.0
```

Work backwards from `L`, one edge at a time. At each step you need only the
*local* derivative of that operation, multiplied by the gradient already
computed for its output:

- `L = q * 2.0`, so `dL/dq` = ?
- `q = p + x`, and a `+` node passes gradient through unchanged, so `dL/dp` = ?
- `p = x * y`, so `dL/dy` = `dL/dp` times the local derivative `dp/dy` = ?

`x` is the interesting one. It reaches `L` by **two different routes** — once
through `p`, and once directly through `q`. When a value influences the output
along several paths, the contributions **add**. That is the multivariable chain
rule, and it is the reason the real engine will use `+=` rather than `=`.

**Exercise.** Fill in the four numbers below by hand. No code, no calculator
tricks — reason it out edge by edge. The check verifies them against your
`numerical_grad` from Section 1.

In [ ]:
dL_dq = None   # YOUR ANSWER
dL_dp = None   # YOUR ANSWER
dL_dy = None   # YOUR ANSWER
dL_dx = None   # YOUR ANSWER  (remember: two paths, add them)

In [ ]:
# --- CHECK 4 ---
def _L(x, y):
    p = x * y
    q = p + x
    return q * 2.0

_num = numerical_grad(_L, [3.0, -2.0])
assert dL_dq is not None and dL_dp is not None, "fill in all four answers"
assert abs(dL_dq - 2.0) < 1e-6, f"dL/dq should be 2.0, got {dL_dq}"
assert abs(dL_dp - 2.0) < 1e-6, f"dL/dp should be 2.0, got {dL_dp}"
assert abs(dL_dx - _num[0]) < 1e-4, f"dL/dx should be {_num[0]:.4f}, got {dL_dx}"
assert abs(dL_dy - _num[1]) < 1e-4, f"dL/dy should be {_num[1]:.4f}, got {dL_dy}"
print("PASS - you just ran backpropagation with your own brain")

---
## 5. `_backward` — teaching each node its own rule

Now automate exactly what you just did by hand. The trick: **no node needs to
know about the network.** Each one only needs two things:

1. its own local derivative (for `c = a*b`, that's `dc/da = b`), and
2. the gradient already sitting on its output (`c.grad`).

Multiply them and add the result into the input's `.grad`. One hop.

So when an operation builds its output Value, it also attaches a small function
describing how to push gradient one step backwards. For addition, the local
derivative is `1` for both inputs, so gradient passes straight through. For
multiplication, each input's local derivative is *the other input's data*.

**Use `+=`, never `=`.** A value used in two places receives two contributions,
and they must accumulate — Section 4 showed you exactly why.

**Exercise.** In THE VALUE CLASS, extend `__add__` and `__mul__` so that each
one defines a local `_backward` function and attaches it to the output Value
before returning it.

In [ ]:
# --- CHECK 5 ---
x = Value(3.0)
y = Value(-2.0)
p = x * y
q = p + x
L = q * Value(2.0)

# manually walk backwards, outermost first
L.grad = 1.0
L._backward()
q._backward()
p._backward()

assert abs(q.grad - 2.0) < 1e-9, f"q.grad should be 2.0, got {q.grad}"
assert abs(p.grad - 2.0) < 1e-9, f"p.grad should be 2.0, got {p.grad}"
assert abs(y.grad - 6.0) < 1e-9, f"y.grad should be 6.0, got {y.grad}"
assert abs(x.grad - (-2.0)) < 1e-9, (
    f"x.grad should be -2.0, got {x.grad}. Two paths reach x - are you using += ?")
print("PASS - each node now knows how to pass gradient backwards")

---
## 6. `backward()` — doing it in the right order, automatically

In the last check *you* chose the order: `L`, then `q`, then `p`. That ordering
matters. A node's gradient is only correct once **every node it feeds into** has
already been processed — because it influences the loss solely through them.

So the rule is: process a node only after all of its consumers are done. That is
a topological ordering of the graph, walked in reverse. (This is the entire
answer to "why do we go backwards" — it is a dependency order, exactly like the
forward pass has one, just mirrored.)

**Exercise.** Add a `backward(self)` method to THE VALUE CLASS that:

1. builds a topological order of the graph — a list where every node appears
   after all the nodes it was built from (a recursive depth-first walk over
   `_prev`, appending each node *after* visiting its children, does this);
2. sets `self.grad = 1.0` (the seed: the derivative of the output w.r.t.
   itself is 1);
3. walks that list **in reverse**, calling `node._backward()` on each.

In [ ]:
# --- CHECK 6 ---
x = Value(3.0); y = Value(-2.0)
L = (x * y + x) * Value(2.0)
L.backward()

_num = numerical_grad(lambda a, b: (a*b + a) * 2.0, [3.0, -2.0])
assert abs(x.grad - _num[0]) < 1e-4, f"x.grad {x.grad} vs numerical {_num[0]}"
assert abs(y.grad - _num[1]) < 1e-4, f"y.grad {y.grad} vs numerical {_num[1]}"

# a deeper graph with heavy reuse
a = Value(1.5); b = Value(-0.5); c = Value(2.0)
d = a * b
e = d + c
f = e * a
g = f + d * c
g.backward()

def _fn(A, B, C):
    D = A * B
    E = D + C
    F = E * A
    return F + D * C

_num = numerical_grad(_fn, [1.5, -0.5, 2.0])
for _name, _v, _n in zip("abc", [a.grad, b.grad, c.grad], _num):
    assert abs(_v - _n) < 1e-4, f"{_name}.grad {_v} vs numerical {_n}"
print("PASS - your engine agrees with numerical gradients. You built autograd.")

---
## 7. More operations

Addition and multiplication alone can only build straight lines. Stacking linear
layers with nothing in between collapses to a single linear layer — depth would
buy you nothing. Networks need a **nonlinearity**.

`tanh` is the classic choice: it squashes any input into `(-1, 1)`, and it has a
famously tidy derivative:

$$\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)$$

Note the derivative is expressed in terms of the *output* — which you have
already computed. That is common and worth noticing.

**Exercise.** Add three methods to THE VALUE CLASS, each following the same
pattern as Section 5 (compute the output, attach a `_backward`, return it):

- `tanh(self)` — use `math.tanh`
- `exp(self)` — derivative of `e^x` is `e^x`, i.e. the output itself
- `__pow__(self, k)` — for a plain-number exponent `k` only; power rule

Then uncomment the two lines at the bottom of the class cell for `__truediv__`
and `__rtruediv__`, which are written in terms of `__pow__`.

In [ ]:
# --- CHECK 7 ---
import math

t = Value(0.7).tanh()
assert abs(t.data - math.tanh(0.7)) < 1e-9, "tanh forward is wrong"

x = Value(0.7)
y = x.tanh()
y.backward()
assert abs(x.grad - (1 - math.tanh(0.7)**2)) < 1e-6, f"tanh backward wrong: {x.grad}"

x = Value(1.3)
y = x.exp()
y.backward()
assert abs(y.data - math.exp(1.3)) < 1e-9, "exp forward is wrong"
assert abs(x.grad - math.exp(1.3)) < 1e-6, f"exp backward wrong: {x.grad}"

x = Value(2.5)
y = x ** 3
y.backward()
assert abs(y.data - 15.625) < 1e-9, "pow forward is wrong"
assert abs(x.grad - 18.75) < 1e-6, f"pow backward wrong: {x.grad}"

a = Value(2.0); b = Value(-3.0)
out = ((a * b + b**2) / a).tanh()
out.backward()
_num = numerical_grad(lambda A, B: math.tanh((A*B + B**2) / A), [2.0, -3.0])
assert abs(a.grad - _num[0]) < 1e-4, f"a.grad {a.grad} vs {_num[0]}"
assert abs(b.grad - _num[1]) < 1e-4, f"b.grad {b.grad} vs {_num[1]}"
print("PASS - nonlinearities, powers and division all differentiate correctly")

---
## 8. A neuron is one line of arithmetic

Everything in a neural network is now within reach of what you have built.

A **neuron** holds a weight per input and one bias. It computes
`w1*x1 + w2*x2 + ... + b`, then applies a nonlinearity. That's it.

A **layer** is a list of neurons all fed the same inputs. An **MLP** is a list of
layers, where each layer's outputs become the next layer's inputs.

**Exercise.** Implement the three classes below. Notes:

- Initialise weights randomly in `[-1, 1]`; initialise bias to `0.0`.
- `parameters()` must return a flat list of every `Value` that should be trained
  — the training loop depends on it.
- Have a `Layer` return a bare `Value` instead of a 1-element list when it has
  only one neuron; it makes the final output easier to work with.

In [ ]:
import random
random.seed(1337)

class Neuron:
    def __init__(self, n_inputs):
        # YOUR CODE HERE: self.w = [...], self.b = ...
        raise NotImplementedError

    def __call__(self, x):
        # YOUR CODE HERE: weighted sum + bias, then .tanh()
        raise NotImplementedError

    def parameters(self):
        # YOUR CODE HERE
        raise NotImplementedError


class Layer:
    def __init__(self, n_inputs, n_neurons):
        raise NotImplementedError

    def __call__(self, x):
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError


class MLP:
    def __init__(self, n_inputs, layer_sizes):
        # e.g. MLP(3, [4, 4, 1]) -> layers of size 4, 4, 1
        raise NotImplementedError

    def __call__(self, x):
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError

In [ ]:
# --- CHECK 8 ---
n = Neuron(3)
out = n([1.0, 2.0, 3.0])
assert isinstance(out, Value), "a neuron should return a Value"
assert -1.0 <= out.data <= 1.0, "tanh output must land in [-1, 1]"
assert len(n.parameters()) == 4, f"3 weights + 1 bias = 4, got {len(n.parameters())}"

lay = Layer(3, 4)
outs = lay([1.0, 2.0, 3.0])
assert len(outs) == 4, f"layer of 4 neurons should give 4 outputs, got {len(outs)}"
assert len(lay.parameters()) == 16, f"4*(3+1) = 16, got {len(lay.parameters())}"

net = MLP(3, [4, 4, 1])
y = net([1.0, 2.0, 3.0])
assert isinstance(y, Value), "final layer has 1 neuron, so return a single Value"
assert len(net.parameters()) == 41, f"expected 41 parameters, got {len(net.parameters())}"
assert all(isinstance(p, Value) for p in net.parameters()), "parameters must be Values"
print(f"PASS - you have a {len(net.parameters())}-parameter neural network")

---
## 9. Training it

The full loop, four steps, repeated:

1. **Forward** — run every input through the net, compare predictions to targets
   with a loss. Use squared error: sum of `(pred - target)**2`.
2. **Zero the gradients** — set `p.grad = 0.0` for every parameter.
3. **Backward** — call `loss.backward()`.
4. **Update** — nudge every parameter *against* its gradient:
   `p.data -= learning_rate * p.grad`.

Step 2 is not optional. Your `_backward` methods use `+=`, so without clearing,
this step's gradients pile on top of last step's and the update is garbage.
Skipping it is the single most common bug in from-scratch implementations, and
it fails *silently* — training just quietly gets worse. (Try deleting it
afterwards and watch.)

Step 4's minus sign is the whole idea of gradient descent: the gradient points
uphill in loss, so you step the other way.

**Exercise.** Write the loop. Train until the loss is below `0.01` — with
`lr = 0.05` this should take well under 200 steps.

In [ ]:
xs = [
    [2.0,  3.0, -1.0],
    [3.0, -1.0,  0.5],
    [0.5,  1.0,  1.0],
    [1.0,  1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

net = MLP(3, [4, 4, 1])
learning_rate = 0.05

for step in range(200):
    # YOUR CODE HERE: forward -> loss -> zero grads -> backward -> update
    # print the loss every 20 steps so you can watch it fall
    pass

preds = [net(x).data for x in xs]
print("predictions:", [f"{p:+.3f}" for p in preds])
print("targets:    ", [f"{y:+.3f}" for y in ys])

In [ ]:
# --- CHECK 9 ---
_loss = sum((net(x).data - y)**2 for x, y in zip(xs, ys))
assert _loss < 0.01, f"final loss is {_loss:.4f}, needs to be under 0.01 - train longer?"
for _p, _y in zip([net(x).data for x in xs], ys):
    assert _p * _y > 0, f"prediction {_p:+.3f} has the wrong sign for target {_y:+.1f}"
print(f"PASS - final loss {_loss:.5f}. You trained a neural network you wrote yourself.")

---
## 10. Two experiments before you go

Quick things to try in a scratch cell — each takes a minute and teaches
something that reading cannot.

1. **Delete `zero_grad`.** Comment out the gradient-clearing step and re-run
   training. Watch the loss explode or stall. That failure mode is worth seeing
   once with your own eyes so you recognise it later.

2. **Crank the learning rate.** Try `lr = 1.0`, then `lr = 0.001`. One diverges,
   one crawls. There is no principled formula for this — everyone tunes it, and
   now you know what both failure directions look like.

---

### What you built

A working reverse-mode automatic differentiation engine, and a neural network
trained with it. PyTorch is this same idea, with tensors instead of scalars and
a lot of engineering — when you write `loss.backward()` there, you now know
precisely what is happening.

### Next

**Notebook 2** ports this to PyTorch and does *behaviour cloning*: you'll train
a network to imitate the scripted controller that lands the rocket, then watch
your own network fly it on the console. That's supervised learning with a
visible payoff — and it sets up everything REINFORCE will need.

Tell me when you've hit `PASS` on Section 9 and I'll build it.